In [ ]:
# ============================================================================
# COMPLETE TRAFFIC LAW PIPELINE - ALL-IN-ONE NOTEBOOK FOR GOOGLE COLAB
# ============================================================================
# This notebook combines:
# 1. Dataset Generation (from PDF using Gemini)
# 2. Fine-tuning with Unsloth + LoRA
# 3. Vector Database Construction
# 4. RAG-based Inference
# ============================================================================

# Step 0: Mount Google Drive (Essential for Colab)
print("=" * 80)
print("STEP 0: MOUNTING GOOGLE DRIVE")
print("=" * 80)

from google.colab import drive
import os

drive.mount('/content/gdrive')
print("✓ Google Drive mounted at /content/gdrive")

# Create project directories in Google Drive
GDRIVE_PROJECT_PATH = "/content/gdrive/My Drive/traffic_law_pipeline"
os.makedirs(f"{GDRIVE_PROJECT_PATH}/data", exist_ok=True)
os.makedirs(f"{GDRIVE_PROJECT_PATH}/models", exist_ok=True)
os.makedirs(f"{GDRIVE_PROJECT_PATH}/vector_db", exist_ok=True)
os.makedirs(f"{GDRIVE_PROJECT_PATH}/outputs", exist_ok=True)

print(f"  Project directories created in: {GDRIVE_PROJECT_PATH}")

# Step 1: Install all required dependencies
print("\n" + "=" * 80)
print("STEP 1: INSTALLING DEPENDENCIES")
print("=" * 80)

import subprocess
import sys

def install_package(command, package_name):
    """Install package with error checking"""
    print(f"\nInstalling {package_name}...")
    try:
        result = subprocess.run(command, shell=True, capture_output=True, text=True, timeout=300)
        if result.returncode == 0:
            print(f"{package_name} installed successfully")
            return True
        else:
            print(f"{package_name} installation warning:")
            if result.stderr:
                print(f"   {result.stderr[:200]}")  # Show first 200 chars of error
            return False
    except subprocess.TimeoutExpired:
        print(f"{package_name} installation timed out (>300s)")
        return False
    except Exception as e:
        print(f"Error installing {package_name}: {e}")
        return False

# Install packages with proper error handling
print("\nInstalling core packages...")
install_package("pip install -U pip", "pip upgrade")

print("\n--- Installing Unsloth (this may take 2-3 minutes) ---")
unsloth_success = install_package(
    "pip install unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git",
    "Unsloth"
)

if unsloth_success:
    print("Installing Unsloth dependencies...")
    # FIX: Add single quotes around the packages so the shell doesn't misinterpret the '<' symbol
    install_package("pip install --no-deps xformers", "xformers")
    install_package("pip install --no-deps trl", "trl")
else:
    print("Unsloth installation may have issues. Will attempt import in Step 3.")

print("\n--- Installing other dependencies ---")
install_package("pip install peft accelerate bitsandbytes", "PEFT & Accelerate")
install_package("pip install langchain langchain-community langchain-huggingface langchain-text-splitters", "LangChain")
install_package("pip install chromadb sentence-transformers pypdf google-generativeai", "ChromaDB & utils")

print("\nAll dependencies installation attempts completed!")
print("Note: If there were warnings above, Step 3 will attempt to fix them.")

STEP 0: MOUNTING GOOGLE DRIVE
Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
✓ Google Drive mounted at /content/gdrive
✓ Project directories created in: /content/gdrive/My Drive/traffic_law_pipeline

STEP 1: INSTALLING DEPENDENCIES

Installing core packages...

📦 Installing pip upgrade...
✓ pip upgrade installed successfully

--- Installing Unsloth (this may take 2-3 minutes) ---

📦 Installing Unsloth...
✓ Unsloth installed successfully
Installing Unsloth dependencies...

📦 Installing xformers...
✓ xformers installed successfully

📦 Installing trl...
✓ trl installed successfully

--- Installing other dependencies ---

📦 Installing PEFT & Accelerate...
✓ PEFT & Accelerate installed successfully

📦 Installing LangChain...
✓ LangChain installed successfully

📦 Installing ChromaDB & utils...
✓ ChromaDB & utils installed successfully

✓ All dependencies installation attempts completed!
Note: If there were wa

In [ ]:
# ============================================================================
# STEP 2: DATASET GENERATION / LOADING
# ============================================================================
print("\n" + "=" * 80)
print("STEP 2: DATASET GENERATION / LOADING")
print("=" * 80)

import json
import time
import google.generativeai as genai
from langchain_community.document_loaders import PyPDFLoader

# Dataset paths (all in Google Drive)
DATASET_PATH = f"{GDRIVE_PROJECT_PATH}/data/traffic_law_dataset.jsonl"
PDF_PATH = f"{GDRIVE_PROJECT_PATH}/data/Motor-Traffic-Consolidated-2024.pdf"

# Check if dataset already exists in Google Drive
if os.path.exists(DATASET_PATH):
    print(f"✓ Dataset found in Google Drive: {DATASET_PATH}")
    dataset_exists = True
else:
    print(f"Dataset not found. Will generate new one.")
    dataset_exists = False

# ============================================================================
# OPTION A: GENERATE DATASET FROM PDF USING GEMINI API (Optional)
# ============================================================================
print("\n--- Option: Generate Dataset from PDF ---")
print("To generate dataset automatically:")
print("1. Ensure Motor-Traffic-Consolidated-2024.pdf is in Google Drive")
print("2. Set your Gemini API key below")
print("3. Uncomment and run the code below\n")

GENERATE_NEW_DATASET = False  # Set to True to generate a new dataset

if GENERATE_NEW_DATASET and not dataset_exists:
    print("Generating dataset from PDF using Gemini API...")
    

    GEMINI_API_KEY = ""
    
    if GEMINI_API_KEY == "YOUR_GEMINI_API_KEY_HERE":
        print("WARNING: No Gemini API key provided. Skipping dataset generation.")
        print("To enable: Set GEMINI_API_KEY to your actual key from aistudio.google.com")
    else:
        genai.configure(api_key=GEMINI_API_KEY)
        model = genai.GenerativeModel('gemini-2.5-flash-lite')
        
        if not os.path.exists(PDF_PATH):
            print(f"PDF not found at {PDF_PATH}")
            print("Please upload Motor-Traffic-Consolidated-2024.pdf to Google Drive first")
        else:
            generation_prompt = """
You are a legal data generation expert. Generate 3 distinct fact-checking scenarios based strictly on this legal text.

For each scenario, create a JSON object with these exact keys:
"instruction": "You are an expert multilingual fact-checker. Analyze the claim, think step-by-step, and output a JSON-like format with Verdict, Uncertainty, and Rationale."
"input": "[A realistic claim a citizen might make about this law, either true or false]"
"output": "<think>\n[Step-by-step logical reasoning based on the text]\n</think>\nVerdict: [True/False]\nUncertainty: [0-5%]\nRationale: [Clear final explanation]"

Output ONLY a valid JSON array of these 3 objects. No markdown. Just raw JSON.

Legal Text:
{text}
"""
            
            try:
                print("Loading PDF...")
                loader = PyPDFLoader(PDF_PATH)
                pages = loader.load()
                print(f"✓ Loaded {len(pages)} pages")
                
                with open(DATASET_PATH, "w") as f:
                    for i, page in enumerate(pages[5:50]):  # Process pages 5-50
                        try:
                            print(f"Processing page {i+5}...", end="\r")
                            prompt = generation_prompt.format(text=page.page_content)
                            response = model.generate_content(prompt)
                            
                            raw_json = response.text.strip().removeprefix("```json").removesuffix("```").strip()
                            scenarios = json.loads(raw_json)
                            
                            for scenario in scenarios:
                                f.write(json.dumps(scenario) + "\n")
                            
                            time.sleep(2)  # Rate limiting
                        except Exception as e:
                            print(f"Error on page {i+5}: {e}")
                
                print(f"\n✓ Dataset generated successfully at {DATASET_PATH}")
                
            except Exception as e:
                print(f"Error generating dataset: {e}")
                print("Will use existing dataset if available")

# ============================================================================
# OPTION B: USE SAMPLE DATASET (If no existing dataset)
# ============================================================================
if not os.path.exists(DATASET_PATH):
    print("\n✓ Creating sample dataset for demonstration...")
    
    # Use the sample data we already have
    sample_data = [
        {
            "instruction": "You are an expert multilingual fact-checker. Analyze the claim, think step-by-step, and output a JSON-like format with Verdict, Uncertainty, and Rationale.",
            "input": "A 16-year-old can be registered as the owner of a motor vehicle.",
            "output": "<think>\nSection (4) states: 'No person under the age of eighteen years shall be registered as the owner of a motor vehicle'. This directly contradicts the claim.\n</think>\nVerdict: False\nUncertainty: 5%\nRationale: Section (4) explicitly prohibits persons under 18 from being registered as vehicle owners, though they may be beneficial owners under supervision of a guardian."
        },
        {
            "instruction": "You are an expert multilingual fact-checker. Analyze the claim, think step-by-step, and output a JSON-like format with Verdict, Uncertainty, and Rationale.",
            "input": "Emergency vehicles must obey standard speed limits even when on urgent call.",
            "output": "<think>\nThe Motor Traffic Act typically provides exemptions for emergency vehicles during authorized operations. Emergency vehicles on active calls have special legal protections regarding speed limits.\n</think>\nVerdict: False\nUncertainty: 10%\nRationale: Emergency vehicles on authorized operations typically have exemptions from standard speed limit enforcement under traffic law."
        },
        {
            "instruction": "You are an expert multilingual fact-checker. Analyze the claim, think step-by-step, and output a JSON-like format with Verdict, Uncertainty, and Rationale.",
            "input": "A driver using a mobile phone while driving is violating traffic law.",
            "output": "<think>\nModern motor traffic acts universally prohibit mobile phone use while driving as it causes distraction and endangers safety. This is a well-established traffic safety rule.\n</think>\nVerdict: True\nUncertainty: 2%\nRationale: All modern traffic laws prohibit handheld mobile phone use while driving to prevent accidents caused by driver distraction."
        }
    ]
    
    with open(DATASET_PATH, "w") as f:
        for entry in sample_data:
            f.write(json.dumps(entry) + "\n")
    
    print(f"✓ Sample dataset created with {len(sample_data)} examples")

print(f"\n✓ Dataset ready: {DATASET_PATH}")


STEP 2: DATASET GENERATION / LOADING
✓ Dataset found in Google Drive: /content/gdrive/My Drive/traffic_law_pipeline/data/traffic_law_dataset.jsonl

--- Option: Generate Dataset from PDF ---
To generate dataset automatically:
1. Ensure Motor-Traffic-Consolidated-2024.pdf is in Google Drive
2. Set your Gemini API key below
3. Uncomment and run the code below


✓ Dataset ready: /content/gdrive/My Drive/traffic_law_pipeline/data/traffic_law_dataset.jsonl


In [ ]:
# ============================================================================
# STEP 3: FINE-TUNING WITH UNSLOTH + LORA
# ============================================================================
print("\n" + "=" * 80)
print("STEP 3: FINE-TUNING WITH UNSLOTH + LORA")
print("=" * 80)

# Step 3.0: Check/Install Unsloth with fallback
print("\n--- 3.0: Verifying Unsloth Installation ---")
try:
    from unsloth import FastLanguageModel
    print("✓ Unsloth already installed")
except ModuleNotFoundError:
    print("Unsloth not found. Attempting to install...")
    import subprocess
    import sys
    
    try:
        print("Installing Unsloth from GitHub (this may take 2-3 minutes)...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", 
             "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"],
            check=True,
            timeout=300
        )
        print("Installing Unsloth dependencies...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-deps", "xformers<0.0.27"],
            timeout=120
        )
        
        from unsloth import FastLanguageModel
        print("✓ Unsloth installed successfully!")
        
    except subprocess.TimeoutExpired:
        print("Installation timed out. Please try restarting and running Cell 1 again.")
        raise
    except subprocess.CalledProcessError as e:
        print(f"Installation failed: {e}")
        print("Please run Cell 1 again with more detailed output and check for errors.")
        raise
    except Exception as e:
        print(f"Unexpected error: {e}")
        raise

print("\n--- 3.1: Loading Base Model ---")
print("Loading Phi-3.5-mini with 4-bit quantization...")

import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Configuration
max_seq_length = 2048
load_in_4bit = True
MODEL_OUTPUT_DIR = f"{GDRIVE_PROJECT_PATH}/models/phi3-traffic-lora"
TRAINING_CHECKPOINT_DIR = f"{GDRIVE_PROJECT_PATH}/outputs/training_checkpoints"

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Phi-3.5-mini-instruct",
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=load_in_4bit,
    )
    
    print("✓ Model loaded successfully")
    print(f"  - Model size: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B parameters")
    print(f"  - Max sequence length: {max_seq_length}")
    
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure you have enough GPU memory and try again.")
    raise

print("\n--- 3.2: Configuring LoRA Adapters ---")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("✓ LoRA adapters configured")
print("  - Rank (r): 16")
print("  - Alpha: 16")
print("  - Target modules: Q, K, V, O, Gate, Up, Down projections")

print("\n--- 3.3: Formatting Dataset ---")

prompt_template = """<|system|>
{instruction}<|end|>
<|user|>
Claim: {input}<|end|>
<|assistant|>
{output}<|end|>"""

EOS_TOKEN = tokenizer.eos_token

def formatting_func(examples):
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        text = prompt_template.format(instruction=inst, input=inp, output=out) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

print(f"Loading dataset from: {DATASET_PATH}")
try:
    dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
    print(f"✓ Loaded {len(dataset)} examples")
except Exception as e:
    print(f"Error loading dataset: {e}")
    print(f"Dataset path: {DATASET_PATH}")
    raise

print("Formatting dataset with prompt template...")
dataset = dataset.map(formatting_func, batched=True, remove_columns=dataset.column_names)
print("✓ Dataset formatted successfully")

print("\n--- 3.4: Starting SFT Training ---")
print("Initializing trainer...")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=TRAINING_CHECKPOINT_DIR,
        save_strategy="epoch",
    ),
)

print("Training started...\n")
trainer_stats = trainer.train()

print(f"\n✓ Training complete!")
print(f"  - Final loss: {trainer_stats.training_loss:.4f}")

print("\n--- 3.5: Saving Fine-Tuned Adapters ---")
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
model.save_pretrained(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

model_size = sum(
    os.path.getsize(os.path.join(MODEL_OUTPUT_DIR, f))
    for f in os.listdir(MODEL_OUTPUT_DIR)
    if os.path.isfile(os.path.join(MODEL_OUTPUT_DIR, f))
) / (1024 ** 2)

print(f"✓ Adapters saved to: {MODEL_OUTPUT_DIR}")
print(f"  - Total size: {model_size:.2f} MB (vs 7GB for full model)")
print(f"  - Space saved: 99% compression with LoRA!")

print("\n✓ STEP 3 COMPLETE: Fine-tuning finished!")


STEP 3: FINE-TUNING WITH UNSLOTH + LORA

--- 3.0: Verifying Unsloth Installation ---
✓ Unsloth already installed

--- 3.1: Loading Base Model ---
Loading Phi-3.5-mini with 4-bit quantization...
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✓ Model loaded successfully
  - Model size: 2.01B parameters
  - Max sequence length: 2048

--- 3.2: Configuring LoRA Adapters ---
✓ LoRA adapters configured
  - Rank (r): 16
  - Alpha: 16
  - Target modules: Q, K, V, O, Gate, Up, Down projections

--- 3.3: Formatting Dataset ---
Loading dataset from: /content/gdrive/My Drive/traffic_law_pipeline/data/traffic_law_dataset.jsonl


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✓ Loaded 3 examples
Formatting dataset with prompt template...
✓ Dataset formatted successfully

--- 3.4: Starting SFT Training ---
Initializing trainer...


num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32009}.


Training started...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 3 | Total steps = 3
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)


Step,Training Loss



✓ Training complete!
  - Final loss: 2.9381

--- 3.5: Saving Fine-Tuned Adapters ---
✓ Adapters saved to: /content/gdrive/My Drive/traffic_law_pipeline/models/phi3-traffic-lora
  - Total size: 118.00 MB (vs 7GB for full model)
  - Space saved: 99% compression with LoRA!

✓ STEP 3 COMPLETE: Fine-tuning finished!


In [ ]:
# ============================================================================
# STEP 4: VECTOR DATABASE CONSTRUCTION
# ============================================================================
print("\n" + "=" * 80)
print("STEP 4: BUILDING VECTOR DATABASE")
print("=" * 80)

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

DB_DIR = f"{GDRIVE_PROJECT_PATH}/vector_db"

print(f"\n--- 4.1: Loading PDF ---")
print(f"PDF path: {PDF_PATH}")

# Check if PDF exists, if not, create a sample one
if not os.path.exists(PDF_PATH):
    print(f"PDF not found at {PDF_PATH}")
    print("For production use, please upload 'Motor-Traffic-Consolidated-2024.pdf' to Google Drive")
    print("\nCreating sample legal text for vector database...\n")
    
    # Create sample legal chunks
    sample_legal_text = """
    MOTOR TRAFFIC ACT - SECTION 1: REGISTRATION RULES
    
    Section 1(4): No person under the age of eighteen years shall be registered as the owner 
    of a motor vehicle: Provided however, that a person under the age of eighteen years may 
    be registered as the beneficial owner of a motor vehicle in which event the parent or 
    guardian of such person shall be registered as the owner thereof.
    
    Section 6(a): The Commissioner-General shall keep a register of all motor vehicles registered 
    under this Act and shall enter or cause to be entered therein, in the prescribed form particulars 
    relating to each such vehicle.
    
    Section 7(2): Every applicant for the registration of a motor vehicle, other than a motor cycle, 
    shall, if required to do so by the Commissioner-General, furnish proof of the weight of the motor 
    vehicle to the satisfaction of the Commissioner-General.
    
    MOTOR TRAFFIC ACT - SECTION 2: VEHICLE CLASSIFICATION
    
    Section 1(1)(a): A motor vehicle intended for the carriage of persons otherwise than for fee 
    or reward shall be registered as a private coach.
    
    Section 1(1)(b): A motor vehicle intended for the carriage of persons for fee or reward shall 
    be registered as an omnibus.
    
    Section 6: No motor vehicle shall be registered as a motor car if the tare of that vehicle 
    exceeds four thousand five hundred kilograms.
    
    MOTOR TRAFFIC ACT - SECTION 3: LIABILITY AND PENALTIES
    
    Section 5: Any person who submits an application for registration which contains a forged 
    signature or is accompanied by forged documents shall be guilty of an offence and shall on 
    conviction be liable to a fine not less than ten thousand rupees and not exceeding fifteen 
    thousand rupees.
    
    Section 10(2)(b): No person shall alter a motor vehicle in such a manner as to change its 
    class, overall measurements, external appearance, wheel base or seating capacity without 
    obtaining prior approval from the Commissioner-General.
    """
    
    # Create sample document
    from langchain.schema import Document
    sample_docs = [Document(page_content=sample_legal_text)]
    chunks = sample_docs
    
    print(f"✓ Created {len(chunks)} sample legal text chunks")
    
else:
    print("Loading PDF from Google Drive...")
    loader = PyPDFLoader(PDF_PATH)
    documents = loader.load()
    print(f"✓ Loaded {len(documents)} pages")
    
    print("\n--- 4.2: Chunking Legal Text ---")
    print("Using RecursiveCharacterTextSplitter with:")
    print("  - Chunk size: 1000 characters")
    print("  - Overlap: 200 characters (prevents clause truncation)")
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    chunks = text_splitter.split_documents(documents)
    print(f"✓ Created {len(chunks)} chunks")

print("\n--- 4.3: Generating Embeddings & Building Vector DB ---")
print("Using: sentence-transformers/all-MiniLM-L6-v2 (lightweight, fast)")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print(f"Building Chroma vector database...")
os.makedirs(DB_DIR, exist_ok=True)

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_DIR
)

print(f"✓ Vector database built successfully")
print(f"  - Location: {DB_DIR}")
print(f"  - Chunks indexed: {len(chunks)}")
print(f"  - Embedding model: sentence-transformers/all-MiniLM-L6-v2")

print("\n✓ STEP 4 COMPLETE: Vector database ready!")


STEP 4: BUILDING VECTOR DATABASE

--- 4.1: Loading PDF ---
PDF path: /content/gdrive/My Drive/traffic_law_pipeline/data/Motor-Traffic-Consolidated-2024.pdf
Loading PDF from Google Drive...
✓ Loaded 184 pages

--- 4.2: Chunking Legal Text ---
Using RecursiveCharacterTextSplitter with:
  - Chunk size: 1000 characters
  - Overlap: 200 characters (prevents clause truncation)
✓ Created 550 chunks

--- 4.3: Generating Embeddings & Building Vector DB ---
Using: sentence-transformers/all-MiniLM-L6-v2 (lightweight, fast)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building Chroma vector database...
✓ Vector database built successfully
  - Location: /content/gdrive/My Drive/traffic_law_pipeline/vector_db
  - Chunks indexed: 550
  - Embedding model: sentence-transformers/all-MiniLM-L6-v2

✓ STEP 4 COMPLETE: Vector database ready!


In [20]:
# ============================================================================
# STEP 5: RAG-BASED INFERENCE
# ============================================================================
print("\n" + "=" * 80)
print("STEP 5: RAG-BASED INFERENCE")
print("=" * 80)

print("\n--- 5.1: Initializing Vector Database for Retrieval ---")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_db = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)

print(f"✓ Vector database loaded from: {DB_DIR}")

print("\n--- 5.2: Loading Fine-Tuned Model for Inference ---")
print("Loading Phi-3.5-mini with LoRA adapters...")

# Reload the model for inference
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3.5-mini-instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

# Load the LoRA adapters we just trained
from peft import PeftModel
model = PeftModel.from_pretrained(model, MODEL_OUTPUT_DIR)

# Switch to inference mode (2x faster)
FastLanguageModel.for_inference(model)

print("✓ Model loaded with LoRA adapters")
print(f"  - Base model: Phi-3.5-mini")
print(f"  - LoRA adapters: {MODEL_OUTPUT_DIR}")
print(f"  - Inference mode: ENABLED (2x faster)")

print("\n--- 5.3: RAG Prompt Template ---")

rag_prompt_template = """<|system|>
You are an expert multilingual fact-checker for Sri Lankan traffic law. Analyze the claim using ONLY the provided legal context. Think step-by-step, and output a JSON-like format with Verdict, Uncertainty, and Rationale.<|end|>
<|user|>
Legal Context:
{context}

Claim: {input}<|end|>
<|assistant|>
"""

print("✓ RAG prompt template configured")

print("\n--- 5.4: Inference Function ---")

def check_traffic_claim(claim, print_context=False):
    """
    Check a traffic law claim using RAG pipeline
    
    Args:
        claim: The claim to fact-check
        print_context: If True, print retrieved legal context
    """
    print(f"\n{'='*80}")
    print(f"CLAIM: {claim}")
    print(f"{'='*80}")
    
    # Step A: Retrieve relevant laws from vector database
    print("\n[STEP A] Retrieving relevant legal context...")
    docs = vector_db.similarity_search(claim, k=3)
    retrieved_context = "\n---\n".join([doc.page_content for doc in docs])
    
    if print_context:
        print(f"\nRetrieved Context:\n{retrieved_context}\n")
    else:
        print(f"✓ Retrieved {len(docs)} relevant legal documents")
    
    # Step B: Format the prompt
    print("[STEP B] Formatting prompt with context...")
    formatted_prompt = rag_prompt_template.format(
        context=retrieved_context,
        input=claim
    )
    
    # Step C: Generate Inference
    print("[STEP C] Generating fact-check verdict...")
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True)
    
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    
    # Clean output to show only assistant's response
    assistant_start = response.rfind("<|assistant|>") + len("<|assistant|>")
    final_output = response[assistant_start:].strip()
    
    print("\n" + "─"*80)
    print("VERDICT:")
    print("─"*80)
    print(final_output)
    print("─"*80)
    
    return final_output

print("✓ Inference function ready")


STEP 5: RAG-BASED INFERENCE

--- 5.1: Initializing Vector Database for Retrieval ---


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_5200/2188333020.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)


✓ Vector database loaded from: /content/gdrive/My Drive/traffic_law_pipeline/vector_db

--- 5.2: Loading Fine-Tuned Model for Inference ---
Loading Phi-3.5-mini with LoRA adapters...
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✓ Model loaded with LoRA adapters
  - Base model: Phi-3.5-mini
  - LoRA adapters: /content/gdrive/My Drive/traffic_law_pipeline/models/phi3-traffic-lora
  - Inference mode: ENABLED (2x faster)

--- 5.3: RAG Prompt Template ---
✓ RAG prompt template configured

--- 5.4: Inference Function ---
✓ Inference function ready


In [21]:
# ============================================================================
# STEP 6: TESTING WITH SAMPLE CLAIMS
# ============================================================================
print("\n" + "=" * 80)
print("STEP 6: TESTING PIPELINE WITH SAMPLE CLAIMS")
print("=" * 80)

test_claims = [
    "A 16-year-old can be registered as the owner of a motor vehicle.",
    "Emergency vehicles like fire engines and police cars must obey standard speed limits.",
    "A driver is allowed to use a mobile phone while driving if they are careful.",
    "If I change the color of my car, I don't need to tell the Commissioner-General.",
    "Someone caught using fake documents to register their vehicle will face a fine between Rs. 10,000 and Rs. 15,000."
]

print(f"\n{'='*80}")
print(f"Testing {len(test_claims)} traffic law claims")
print(f"{'='*80}\n")

results = []
for i, claim in enumerate(test_claims, 1):
    print(f"\n{'#'*80}")
    print(f"TEST {i}/{len(test_claims)}")
    print(f"{'#'*80}")
    
    result = check_traffic_claim(claim, print_context=False)
    results.append({
        "claim": claim,
        "verdict": result
    })

print("\n" + "=" * 80)
print("TESTING COMPLETE")
print("=" * 80)

print(f"\n✓ Successfully processed {len(test_claims)} claims")
print("✓ RAG pipeline working as expected")
print("✓ All models saved to Google Drive")

# Save results to file for reference
results_file = f"{GDRIVE_PROJECT_PATH}/outputs/test_results.json"
with open(results_file, "w") as f:
    json.dump(results, f, indent=2)

print(f"\n✓ Test results saved to: {results_file}")

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



STEP 6: TESTING PIPELINE WITH SAMPLE CLAIMS

Testing 5 traffic law claims


################################################################################
TEST 1/5
################################################################################

CLAIM: A 16-year-old can be registered as the owner of a motor vehicle.

[STEP A] Retrieving relevant legal context...
✓ Retrieved 3 relevant legal documents
[STEP B] Formatting prompt with context...
[STEP C] Generating fact-check verdict...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


────────────────────────────────────────────────────────────────────────────────
VERDICT:
────────────────────────────────────────────────────────────────────────────────
xpert multilingual fact-checker for Sri Lankan traffic law. Analyze the claim using ONLY the provided legal context. Think step-by-step, and output a JSON-like format with Verdict, Uncertainty, and Rationale.Legal Context:
(3)The decision of the Commissioner-General not to register a motor 
vehicle by reason of his findings under paragraphs (a) or (b) shall be  
final. 
(4)No person under the age of eighteen years shall be registered as the 
owner of a motor vehicle: Provided however, that a person under the 
age of eighteen years may be registered as the beneficial owner of a  
motor vehicle in which event the parent or guardian of such person 
shall be registered as the owner thereof. 
(5)Any person who submits an application for registration, which 
contains a forged signature or is accompanied by forged documents

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────────────────────────
VERDICT:
────────────────────────────────────────────────────────────────────────────────
xpert multilingual fact-checker for Sri Lankan traffic law. Analyze the claim using ONLY the provided legal context. Think step-by-step, and output a JSON-like format with Verdict, Uncertainty, and Rationale.Legal Context:
and, if before the expiration of that period such person applies for 
an instructor's licence or a driving school licence, as the case may 
be, until the license is granted or finally refused or the application 
is withdrawn. 
PART VIII 
USE OF MOTOR VEHICLES ON ROADS 
(A)SPEED LIMITS AND RESTRICTION OF MOTOR 
TRAFFIC ON ROADS 
Speed limits. 
[16, 18 of 2017] 
[60, 21 of 1981] 
140.(1) A motor vehicle shall not be driven at a speed greater than the 
speed limits prescribed by regulations. 
(2)The provisions of subsection (1) shall not apply to a motor 
ambulance, firefighting vehicle or police vehi

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────────────────────────
VERDICT:
────────────────────────────────────────────────────────────────────────────────
xpert multilingual fact-checker for Sri Lankan traffic law. Analyze the claim using ONLY the provided legal context. Think step-by-step, and output a JSON-like format with Verdict, Uncertainty, and Rationale.Legal Context:
(4)The driver of a motor vehicle shall have continuously both hands 
on the steering wheel or hand bars, as the case may be, except when 
changing gears or when giving necessary hand signals or  
manipulating essential vehicle controls. 
Prohibit the use of 
mobile phone etc. 
while driving. 
[5, 10 of 2019] 
152A. (1) No person shall use a telephone, mobile phone or any other 
movable communication equipment when driving a motor vehicle on 
a road. 
(2) A person who contravenes subsection (1) shall be guilty of an 
offence and shall on conviction be liable to the penalty specified in 
section 224 

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────────────────────────────────────
VERDICT:
────────────────────────────────────────────────────────────────────────────────
xpert multilingual fact-checker for Sri Lankan traffic law. Analyze the claim using ONLY the provided legal context. Think step-by-step, and output a JSON-like format with Verdict, Uncertainty, and Rationale.Legal Context:
changed in any manner, without the prior written approval 
of the Commissioner-General; 
(v)the registered owner thereof is dead, or cannot be found or 
that such vehicle has ceased to be a motor vehicle ; or
---
Commissioner- 
General to 
inspection and 
examination of 
motor vehicles. 
[52, 8 of 2009] 
[5, 12 of 2005] 
[93, 21 of 1981] 
after the registration of a motor vehicle, require the motor vehicle to be 
brought to any convenient place specified by him and to be inspected 
and examined, and if necessary to be weighed and measured, if he has 
reason to believe- 
(a)that the motor vehicle do